In [1]:
!pip install pandas numpy scikit-learn streamlit requests matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 89.3 MB/s eta 0:00:00


In [22]:
import pandas as pd

movies = pd.read_csv("movie.csv")
ratings = pd.read_csv("rating.csv")
genome_scores = pd.read_csv("genome_scores.csv")
genome_tags = pd.read_csv("genome_tags.csv")

print(movies.shape, ratings.shape, genome_scores.shape)

(27278, 3) (20000263, 4) (11709768, 3)


In [23]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [28]:
movies['genres'] = movies['genres'].str.replace('|', ' ')
movies['title_clean'] = movies['title'].str.replace(r"\(\d{4}\)", "", regex=True).str.strip()

# keep only strong signal movies
counts = ratings.groupby('movieId').size().reset_index(name='count')
movies = movies[movies['movieId'].isin(counts[counts['count'] > 50]['movieId'])]

print("Filtered:", movies.shape)

Filtered: (10473, 7)


In [29]:
pop = ratings.groupby('movieId').agg({'rating':['count','mean']}).reset_index()
pop.columns = ['movieId','rating_count','rating_mean']

movies = movies.merge(pop, on='movieId')

In [30]:
pop = ratings.groupby('movieId').agg({'rating':['count','mean']}).reset_index()
pop.columns = ['movieId','rating_count','rating_mean']

movies = movies.merge(pop, on='movieId')

In [31]:
# merge genome
genome = genome_scores.merge(genome_tags, on='tagId')

# keep top 5 tags per movie (VERY IMPORTANT)
genome = genome.sort_values(['movieId','relevance'], ascending=[True, False])
genome = genome.groupby('movieId').head(5)

# convert to text
genome_features = genome.groupby('movieId')['tag'].apply(lambda x: " ".join(x)).reset_index()

movies = movies.merge(genome_features, on='movieId', how='left')
movies['tag'] = movies['tag'].fillna('')

In [32]:
movies['combined'] = (
    movies['genres'] + " " +
    movies['tag'] + " " +
    movies['rating_mean'].astype(str)
)

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=2000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['combined'])

In [34]:
from sklearn.metrics.pairwise import linear_kernel

indices = pd.Series(movies.index, index=movies['title_clean']).drop_duplicates()

def recommend(title, k=10):
    if title not in indices:
        return pd.DataFrame()

    idx = indices[title]

    sim = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()
    sim_idx = sim.argsort()[-k-1:-1][::-1]

    recs = movies.iloc[sim_idx]
    return recs.sort_values(by=['rating_mean','rating_count'], ascending=False)

In [35]:
import numpy as np

ratings['relevant'] = ratings['rating'] >= 4

user_likes = ratings.groupby('userId').apply(
    lambda x: set(x[x['relevant']]['movieId'])
).to_dict()

def precision_at_k(rec, rel, k):
    return len(set(rec[:k]) & rel) / k

def recall_at_k(rec, rel, k):
    return len(set(rec[:k]) & rel) / len(rel) if rel else 0

def average_precision(rec, rel, k):
    score = 0
    hits = 0
    for i in range(k):
        if rec[i] in rel:
            hits += 1
            score += hits / (i+1)
    return score / min(len(rel), k)

def ndcg_at_k(rec, rel, k):
    dcg = sum([(1/np.log2(i+2)) for i in range(k) if rec[i] in rel])
    idcg = sum([1/np.log2(i+2) for i in range(min(len(rel), k))])
    return dcg/idcg if idcg>0 else 0

/tmp/ipykernel_1035/2443380227.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_likes = ratings.groupby('userId').apply(


In [36]:
import matplotlib.pyplot as plt
import seaborn as sns

results = {"precision":[], "recall":[], "map":[], "ndcg":[]}

users = list(user_likes.keys())[:50]

for user in users:
    liked = user_likes[user]
    if not liked: continue

    test_movie = list(liked)[0]
    row = movies[movies['movieId']==test_movie]
    if row.empty: continue

    title = row['title_clean'].values[0]
    recs = recommend(title, 10)

    rec_ids = list(recs['movieId'])

    results["precision"].append(precision_at_k(rec_ids, liked, 10))
    results["recall"].append(recall_at_k(rec_ids, liked, 10))
    results["map"].append(average_precision(rec_ids, liked, 10))
    results["ndcg"].append(ndcg_at_k(rec_ids, liked, 10))

avg = {k:np.mean(v) for k,v in results.items()}
print(avg)

# GRAPH
plt.figure()
plt.bar(avg.keys(), avg.values())
plt.title("Model Performance")
plt.ylim(0,1)
plt.show()

IndexError: positional indexers are out-of-bounds

In [ ]:
app_code = """
import streamlit as st
import pandas as pd
import requests
from sklearn.metrics.pairwise import linear_kernel
from sklearn.feature_extraction.text import TfidfVectorizer

API_KEY = "YOUR_OMDB_API_KEY"

st.set_page_config(layout="wide")

# ---------- LOAD ----------
@st.cache_data
def load():
    movies = pd.read_csv("ml-latest/movies.csv")
    ratings = pd.read_csv("ml-latest/ratings.csv")

    movies['genres'] = movies['genres'].str.replace('|', ' ')
    movies['title_clean'] = movies['title'].str.replace(r"\\(\\d{4}\\)", "", regex=True).str.strip()

    counts = ratings.groupby('movieId').size().reset_index(name='count')
    movies = movies[movies['movieId'].isin(counts[counts['count'] > 50]['movieId'])]

    return movies

movies = load()

# MODEL
@st.cache_resource
def model(data):
    tfidf = TfidfVectorizer(max_features=2000)
    return tfidf.fit_transform(data['genres'])

tfidf_matrix = model(movies)

indices = pd.Series(movies.index, index=movies['title_clean']).drop_duplicates()

def fetch(title):
    try:
        data = requests.get(f"http://www.omdbapi.com/?t={title}&apikey={API_KEY}").json()
        return data.get("Poster"), data.get("Plot"), data.get("Year")
    except:
        return None,None,None

def rec(title):
    if title not in indices: return []
    idx = indices[title]
    sim = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()
    ids = sim.argsort()[-11:-1][::-1]
    return movies.iloc[ids]

# UI
st.markdown("<h1 style='text-align:center'>🎬 MOVIE VILLAGE</h1>", unsafe_allow_html=True)

search = st.text_input("Search Movie")

if search:
    p,plot,year = fetch(search)
    col1,col2 = st.columns([1,2])

    with col1:
        if p: st.image(p)

    with col2:
        st.write(year)
        st.write(plot)

    st.subheader("Recommendations")

    r = rec(search)
    cols = st.columns(5)

    for i,row in enumerate(r.itertuples()):
        with cols[i%5]:
            p,_,_ = fetch(row.title_clean)
            if p: st.image(p)
            st.caption(row.title_clean)

else:
    st.subheader("Trending")

    top = movies.head(12)
    cols = st.columns(6)

    for i,row in enumerate(top.itertuples()):
        with cols[i%6]:
            p,_,_ = fetch(row.title_clean)
            if p: st.image(p)
            st.caption(row.title_clean)
"""

with open("app.py","w") as f:
    f.write(app_code)